In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

df = spark.read.parquet("/Volumes/workspace/default/fema_claims_project/fema_raw.parquet/")

# Type casting (.withColumn("Income", col("Income").cast("double")))
numeric_cols = ["amountPaidOnBuildingClaim", "amountPaidOnContentsClaim",
    "totalBuildingInsuranceCoverage", "totalContentsInsuranceCoverage",
    "amountPaidOnIncreasedCostOfComplianceClaim"]
int_cols = ["yearOfLoss", "numberOfFloorsInTheInsuredBuilding"]

#Cast text strings to proper numbers. "50000" → 50000.0
for c in numeric_cols:
    df = df.withColumn(c, F.col(c).cast(DoubleType()))
for c in int_cols:
    df = df.withColumn(c, F.col(c).cast(IntegerType()))


# Null imputation (df.na.fill({"Income": mean_income}))
df = df.na.fill({c: 0.0 for c in numeric_cols})
df = df.na.fill({"numberOfFloorsInTheInsuredBuilding": 1})

# Data validation (Rubric requirement)
df = df.filter(
    (F.col("amountPaidOnBuildingClaim") >= 0) &
    (F.col("totalBuildingInsuranceCoverage") > 0) &
    (F.col("yearOfLoss").between(1970, 2025)) &
    (F.col("state").isNotNull()))
print(f"✅ Cleaned: {df.count():,} rows")

✅ Cleaned: 2,641,236 rows


In [0]:
# ═══════════════════════════════════════════════════════════
# CELL 2: BROADCAST JOIN + FEATURE ENGINEERING
# Rubric: "Broadcast join implementation where appropriate" ✅
# ═══════════════════════════════════════════════════════════

# Create small lookup table for state risk tiers
high_risk = [("FL",3),("TX",3),("LA",3),("NJ",2),("NY",2),("SC",2),("NC",2),("CA",2)]
state_risk_df = spark.createDataFrame(high_risk, ["state_code", "risk_tier"])


# BROADCAST JOIN — small table sent to every worker (no shuffle!)
df = df.join(F.broadcast(state_risk_df),
    df["state"] == state_risk_df["state_code"], "left")
df = df.na.fill({"risk_tier": 1})


# Feature engineering (10+ new columns)
coastal = ["FL","TX","LA","NJ","NY","SC","NC","CA","AL","MS"]
df = (df
    .withColumn("high_payout", F.when(F.col("amountPaidOnBuildingClaim")>50000,1).otherwise(0))
    .withColumn("total_paid", F.col("amountPaidOnBuildingClaim")+F.col("amountPaidOnContentsClaim"))
    .withColumn("claim_ratio",
        F.when(F.col("totalBuildingInsuranceCoverage")>0,
            F.col("amountPaidOnBuildingClaim")/F.col("totalBuildingInsuranceCoverage"))
        .otherwise(0.0))
    .withColumn("claim_age", F.lit(2025)-F.col("yearOfLoss"))
    .withColumn("is_coastal", F.when(F.col("state").isin(coastal),1).otherwise(0))
    .withColumn("has_contents_claim", F.when(F.col("amountPaidOnContentsClaim")>0,1).otherwise(0))
    .withColumn("has_icc", F.when(F.col("amountPaidOnIncreasedCostOfComplianceClaim")>0,1).otherwise(0))
    .withColumn("decade", (F.floor(F.col("yearOfLoss")/10)*10).cast("int"))
    .withColumn("coverage_tier",
        F.when(F.col("totalBuildingInsuranceCoverage")<100000,0)
         .when(F.col("totalBuildingInsuranceCoverage")<200000,1).otherwise(2))
    .withColumn("payout_per_floor",
        F.when(F.col("numberOfFloorsInTheInsuredBuilding")>0,
            F.col("amountPaidOnBuildingClaim")/F.col("numberOfFloorsInTheInsuredBuilding"))
        .otherwise(F.col("amountPaidOnBuildingClaim")))
    .withColumn("log_building_claim", F.log1p(F.col("amountPaidOnBuildingClaim")))
)

In [0]:
# Bar chart of top 10 states by average payout
top_states = (df.groupBy("state")
    .agg(F.avg("amountPaidOnBuildingClaim").alias("avg_payout"))
    .sort("avg_payout", ascending=False)
    .limit(10))
display(top_states, "bar", {"x": "state", "y": "avg_payout"})

state,avg_payout
MS,38206.57242303512
FL,38137.34244620824
TX,34886.71734701921
LA,34826.144632206466
NY,29132.533696200542
VT,28774.909007739083
NJ,27242.738915458143
TN,23022.594480893385
HI,22627.533046701257
AL,22249.70896522439


Databricks visualization. Run in Databricks to view.

In [0]:
# ═══════════════════════════════════════════════════════════
# CELL 3: DISTRIBUTED EDA + CORRELATION
# Dr Stamou: df.describe().show()
#            df.stat.corr(col_name, "label")
# ═══════════════════════════════════════════════════════════

# 1. Descriptive stats
display(df.select("amountPaidOnBuildingClaim","total_paid","claim_ratio","yearOfLoss").describe())

# 2. Correlation with target ( Correlation = how strongly two things are related. +1 = perfect positive, 0 = none, -1 = perfect negative.)
target = "amountPaidOnBuildingClaim"
feature_cols = ["totalBuildingInsuranceCoverage","claim_ratio","claim_age",
    "is_coastal","has_contents_claim","has_icc","payout_per_floor","risk_tier"]
for c in feature_cols:
    corr = df.stat.corr(c, target)
    print(f"  Corr({c}, {target}): {corr:.4f}")


# 3. Correlation matrix for Tableau heatmap
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
corr_assembler = VectorAssembler(inputCols=feature_cols, outputCol="corr_features", handleInvalid="skip")
corr_df = corr_assembler.transform(df.sample(withReplacement=False, fraction=1.0, seed=42))
corr_matrix = Correlation.corr(corr_df, "corr_features").head()[0]
print("Correlation matrix computed ✅")

# 4. Class distribution
display(df.groupBy("high_payout").count())

# 5. Group by state 
display(df.groupBy("state").agg(
    F.avg("amountPaidOnBuildingClaim").alias("avg_payout"),
    F.count("*").alias("count"))
    .sort("avg_payout", ascending=False).limit(10))

# 6. Persist + Save partitioned Silver Parquet
# df.persist() - Available in the Enterprise edition

# Partitioned by state → queries filtering by state skip 49 other folders.
df.write.mode("overwrite").partitionBy("state").parquet("/Volumes/workspace/default/fema_claims_project/fema_silver")


print("✅ Silver Parquet saved!")

summary,amountPaidOnBuildingClaim,total_paid,claim_ratio,yearOfLoss
count,2641236,2641236,2641236,2641236
mean,27573.11053477617,33062.77940583101,0.2126703722580053,2003.242873790907
stddev,67388.01397949451,76778.16375198968,1.540183938470842,12.878392847491561
min,0.0,-5000.0,0.0,1978
max,1.074147693E7,1.084147693E7,1424.9539000000002,2025


  Corr(totalBuildingInsuranceCoverage, amountPaidOnBuildingClaim): 0.1402
  Corr(claim_ratio, amountPaidOnBuildingClaim): 0.1016
  Corr(claim_age, amountPaidOnBuildingClaim): -0.2560
  Corr(is_coastal, amountPaidOnBuildingClaim): 0.1194
  Corr(has_contents_claim, amountPaidOnBuildingClaim): 0.1974
  Corr(has_icc, amountPaidOnBuildingClaim): 0.1070
  Corr(payout_per_floor, amountPaidOnBuildingClaim): 0.8827
  Corr(risk_tier, amountPaidOnBuildingClaim): 0.1281
Correlation matrix computed ✅


high_payout,count
0,2190857
1,450379


state,avg_payout,count
MS,38206.57242303512,62496
FL,38137.34244620824,434918
TX,34886.71734701921,383904
LA,34826.144632206466,475253
NY,29132.533696200542,170448
VT,28774.909007739083,3618
NJ,27242.738915458143,197346
TN,23022.594480893385,17193
HI,22627.533046701257,5396
AL,22249.70896522439,43565


✅ Silver Parquet saved!


In [0]:
# ═══════════════════════════════════════════════════════════
# CELL 4: CUSTOM TRANSFORMER + FEATURE PIPELINE
# Rubric: "Custom transformer for domain-specific features" ✅
#  VectorAssembler → StandardScaler pipeline
# ═══════════════════════════════════════════════════════════

from pyspark.ml import Pipeline, Transformer
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable
from pyspark.ml.feature import VectorAssembler, StandardScaler

# CUSTOM TRANSFORMER — a machine we invented! Combines 4 signals into 1 risk score
class FloodRiskScorer(Transformer, DefaultParamsReadable, DefaultParamsWritable):
    """Domain-specific composite risk score."""
    def __init__(self): super().__init__(); self.uid = "FloodRiskScorer"
    def _transform(self, df):
        return df.withColumn("flood_risk_score",
            (F.col("claim_ratio").cast("double")*0.4) +
            (F.col("is_coastal").cast("double")*0.3) +
            (F.col("has_contents_claim").cast("double")*0.2) +
            (F.col("has_icc").cast("double")*0.1))


ml_features = ["totalBuildingInsuranceCoverage","yearOfLoss",
    "numberOfFloorsInTheInsuredBuilding","claim_ratio","claim_age",
    "is_coastal","has_contents_claim","has_icc","coverage_tier",
    "payout_per_floor","amountPaidOnContentsClaim","risk_tier","flood_risk_score"]

risk_scorer = FloodRiskScorer()
assembler = VectorAssembler(inputCols=ml_features, outputCol="raw_features", handleInvalid="skip")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)
prep_pipeline = Pipeline(stages=[risk_scorer, assembler, scaler])

# Split data
train_df, val_df, test_df = df.randomSplit([0.7, 0.15, 0.15], seed=42)
prep_model = prep_pipeline.fit(train_df)
train_p = prep_model.transform(train_df)
val_p = prep_model.transform(val_df)
test_p = prep_model.transform(test_df)

# Save prepared data as Parquet for other notebooks
train_p.write.mode("overwrite").parquet("/Volumes/workspace/default/fema_claims_project/train_prepared")
val_p.write.mode("overwrite").parquet("/Volumes/workspace/default/fema_claims_project/val_prepared")
test_p.write.mode("overwrite").parquet("/Volumes/workspace/default/fema_claims_project/test_prepared")
print(f"✅ Train: {train_p.count():,} | Val: {val_p.count():,} | Test: {test_p.count():,}")

✅ Train: 1,849,377 | Val: 395,810 | Test: 396,049
